In [33]:
'''
============================================
RFM 客户价值分析 —— 完整流程
============================================

RFM 是衡量客户价值的三个核心指标：
  R（Recency）  ：最近一次消费时间 → 越小越好（离现在越近越活跃）
  F（Frequency） ：消费频率         → 越大越好（消费次数越多越忠诚）
  M（Monetary）  ：消费金额         → 越大越好（花钱越多越有价值）

本 notebook 使用一家公司 2015~2018 年的销售数据，计算每年每个会员的 RFM 值，
然后进行等级划分，最后用 3D 柱状图展示不同 RFM 等级的客户分布。

步骤概览：
  1. 数据加载      - 从 sales.xlsx 读取各年份数据
  2. 数据预处理    - 清洗、过滤、计算时间节点
  3. 数据合并      - 将各年份数据合并
  4. 特征工程      - 提取年份、计算消费时间差
  5. RFM 聚合      - 按会员 ID 和年份聚合
  6. 等级划分      - 对 R/F/M 打分
  7. 结果导出      - 保存到 Excel
  8. 3D 可视化     - 用 pyecharts 展示分布
============================================
'''

# ========== 导入所需库 ==========
import pandas as pd  # 数据处理核心库：读取 Excel/CSV、数据透视、分组聚合等
import numpy as np  # 数值计算库：数组运算、数学函数
import matplotlib.pyplot as plt  # 基础绘图库
import seaborn as sns  # 高级统计可视化（基于 matplotlib）
from datetime import datetime, timedelta  # 日期处理

import warnings
warnings.filterwarnings('ignore')  # 忽略警告，保持输出整洁

# 设置 matplotlib 中文显示
plt.rcParams['font.sans-serif'] = ['SimHei']  # 指定中文字体（黑体）
plt.rcParams['axes.unicode_minus'] = False  # 解决负号显示为方块的问题

# pyecharts 交互式图表（可在 notebook 中生成可旋转的 3D 图表）
from pyecharts.charts import Bar3D  # 3D 柱状图
from pyecharts import options as opts  # 图表配置选项


# RFM 客户价值分析
## 基于 2015~2018 年销售数据的会员分层分析

### 什么是 RFM？
| 指标 | 全称 | 含义 | 评分方向 |
|------|------|------|----------|
| **R** | Recency | 最近一次消费距今多少天 | 越小越好（分数越高）|
| **F** | Frequency | 一段时间内消费了多少次 | 越大越好 |
| **M** | Monetary | 一段时间内消费了多少钱 | 越大越好 |

### 分析流程
1. 加载 4 年销售数据 + 会员等级数据
2. 清洗数据，计算每年末的时间节点
3. 按会员 ID 和年份聚合 RFM 值
4. 对 R/F/M 分别评分（1~3分），组合成 RFM_label
5. 用 3D 柱状图展示各类型客户在各年份的分布

In [34]:
'''
============================================
第1步：加载数据
============================================
sales.xlsx 包含了5个工作表（sheet）：
  - '2015', '2016', '2017', '2018'：各年份的订单数据
  - '会员等级'：所有会员的等级信息

pd.read_excel(file, sheet_name=列表) 的用法：
  可以一次读取多个 worksheet，返回一个字典
  字典的键 = sheet 名称，值 = 对应的 DataFrame
============================================
'''

# 定义要读取的 sheet 名称列表
sheet_names = ['2015', '2016', '2017', '2018', '会员等级']

# 一次读取所有 sheet，返回字典 {sheet名: DataFrame}
sheet_dict = pd.read_excel('sales.xlsx', sheet_name=sheet_names)

# sheet_dict['2015'] 就是 2015 年的订单数据，DataFrame 格式
# sheet_dict['会员等级'] 是会员等级数据

print("===== 2015年数据概览 =====")
sheet_dict['2015'].info()
# .info() 显示 DataFrame 的列名、非空数量、数据类型


===== 2015年数据概览 =====
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30774 entries, 0 to 30773
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   会员ID    30774 non-null  int64         
 1   订单号     30774 non-null  int64         
 2   提交日期    30774 non-null  datetime64[ns]
 3   订单金额    30774 non-null  float64       
dtypes: datetime64[ns](1), float64(1), int64(2)
memory usage: 961.8 KB


In [35]:
'''
============================================
第2步：数据预处理
============================================
对每个年份的数据分别进行清洗：
  1. dropna()         — 删除有空缺值的行
  2. 过滤 订单金额 > 1 — 排除金额异常（过小）的订单
  3. 计算 max_year_date — 该年份的最大订单日期，作为"时间节点"
     后面计算 R（Recency）时要用：R = 时间节点 - 实际购买日期
  
  sheet_names[:-1] 表示去掉最后一个元素（'会员等级'），
  因为会员等级表不需要这些清洗操作
============================================
'''

# ===== 清洗各年份订单数据 =====
for i in sheet_names[:-1]:  # 遍历 2015, 2016, 2017, 2018
    sheet_dict[i] = sheet_dict[i].dropna()  # 删除缺失值
    sheet_dict[i] = sheet_dict[i][sheet_dict[i]['订单金额'] > 1]  # 过滤"金额<=1"的异常订单
    sheet_dict[i]['max_year_date'] = sheet_dict[i]['提交日期'].max()  # 当年最大日期作为时间节点

# ===== 打印各年份 + 会员等级的信息 =====
for i in sheet_names:
    print(f"\n===== {i} =====")
    print(sheet_dict[i].info())
    print(sheet_dict[i].head())



===== 2015 =====
<class 'pandas.core.frame.DataFrame'>
Index: 30574 entries, 0 to 30773
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   会员ID           30574 non-null  int64         
 1   订单号            30574 non-null  int64         
 2   提交日期           30574 non-null  datetime64[ns]
 3   订单金额           30574 non-null  float64       
 4   max_year_date  30574 non-null  datetime64[ns]
dtypes: datetime64[ns](2), float64(1), int64(2)
memory usage: 1.4 MB
None
          会员ID         订单号       提交日期    订单金额 max_year_date
0  15278002468  3000304681 2015-01-01   499.0    2015-12-31
1  39236378972  3000305791 2015-01-01  2588.0    2015-12-31
2  38722039578  3000641787 2015-01-01   498.0    2015-12-31
3  11049640063  3000798913 2015-01-01  1572.0    2015-12-31
4  35038752292  3000821546 2015-01-01    10.1    2015-12-31

===== 2016 =====
<class 'pandas.core.frame.DataFrame'>
Index: 41001 entries, 0 to 41277

In [36]:
'''
============================================
第3步：合并数据
============================================
把 2015~2018 四年的数据合并成一个大的 DataFrame。

pd.concat([df1, df2, ...], ignore_index=True) 的用法：
  将多个 DataFrame 按行堆叠起来
  ignore_index=True 表示忽略原有的行号，重新从 0 开始编号
============================================
'''

# 合并所有年份（去掉 '会员等级' 表）
data = pd.concat([sheet_dict[i] for i in sheet_names[:-1]], ignore_index=True)

print(f"合并后的数据共 {len(data)} 行, {len(data.columns)} 列")
print(data)


合并后的数据共 202827 行, 5 列
               会员ID         订单号       提交日期    订单金额 max_year_date
0       15278002468  3000304681 2015-01-01   499.0    2015-12-31
1       39236378972  3000305791 2015-01-01  2588.0    2015-12-31
2       38722039578  3000641787 2015-01-01   498.0    2015-12-31
3       11049640063  3000798913 2015-01-01  1572.0    2015-12-31
4       35038752292  3000821546 2015-01-01    10.1    2015-12-31
...             ...         ...        ...     ...           ...
202822  39229485704  4354225182 2018-12-31   249.0    2018-12-31
202823  39229021075  4354225188 2018-12-31    89.0    2018-12-31
202824  39288976750  4354230034 2018-12-31    48.5    2018-12-31
202825     26772630  4354230163 2018-12-31  3196.0    2018-12-31
202826  39455580335  4354235084 2018-12-31  2999.0    2018-12-31

[202827 rows x 5 columns]


In [37]:
'''
============================================
第4步：新增 year 列
============================================
从 提交日期 中提取年份，方便后续按年份分组分析。

.dt 是 pandas 的"日期时间访问器"（datetime accessor），
专门用来操作 datetime64 类型的列。
.dt.year 可以提取年份，类似的还有 .dt.month, .dt.day 等
============================================
'''

# 从提交日期中提取年份
data['year'] = data['提交日期'].dt.year

print("===== 添加 year 列后的数据 =====")
print(data.head())


===== 添加 year 列后的数据 =====
          会员ID         订单号       提交日期    订单金额 max_year_date  year
0  15278002468  3000304681 2015-01-01   499.0    2015-12-31  2015
1  39236378972  3000305791 2015-01-01  2588.0    2015-12-31  2015
2  38722039578  3000641787 2015-01-01   498.0    2015-12-31  2015
3  11049640063  3000798913 2015-01-01  1572.0    2015-12-31  2015
4  35038752292  3000821546 2015-01-01    10.1    2015-12-31  2015


In [38]:
'''
============================================
第5步：计算 Recency（R 值）
============================================
R（Recency）表示"最近一次消费距今多少天"。
天数越少 → 最近刚买过 → 越活跃

计算公式：
  R = max_year_date（当年最后一天） - 提交日期（实际购买日期）
  差值用 .dt.days 转为天数

例如 2015-01-01 的订单：
  R = 2015-12-31 - 2015-01-01 = 364 天
  表示这名会员在 2015 年 1月1日 下的单，到年底已经过了 364 天
============================================
'''

# 计算每个订单的 "购买时间距离年底的天数"
data['date_diff'] = (data['max_year_date'] - data['提交日期']).dt.days
# .dt.days 将 timedelta（时间差）转为整数天数

print("===== 时间差验证 =====")
print(data['max_year_date'] - data['提交日期'])

print("\n===== 添加 date_diff 列后的数据 =====")
print(data.head())


===== 时间差验证 =====
0        364 days
1        364 days
2        364 days
3        364 days
4        364 days
           ...   
202822     0 days
202823     0 days
202824     0 days
202825     0 days
202826     0 days
Length: 202827, dtype: timedelta64[ns]

===== 添加 date_diff 列后的数据 =====
          会员ID         订单号       提交日期    订单金额 max_year_date  year  date_diff
0  15278002468  3000304681 2015-01-01   499.0    2015-12-31  2015        364
1  39236378972  3000305791 2015-01-01  2588.0    2015-12-31  2015        364
2  38722039578  3000641787 2015-01-01   498.0    2015-12-31  2015        364
3  11049640063  3000798913 2015-01-01  1572.0    2015-12-31  2015        364
4  35038752292  3000821546 2015-01-01    10.1    2015-12-31  2015        364


In [39]:
'''
============================================
第6步：RFM 聚合计算
============================================
按 年份(year) 和 会员ID 分组，计算每个会员每年的 RFM 值：
  R（date_diff 最小值）：取当年所有订单中距离年底最近的那天
      → 该会员当年"最晚一次消费"距年底多少天
  F（订单号 count）    ：统计该会员当年下了多少单
  M（订单金额 sum）    ：统计该会员当年花了多少钱

.groupby(['A', 'B']).agg({...}) 的用法：
  先按 A 和 B 分组
  然后对不同列做不同的聚合操作
  .reset_index() 把分组键从"索引"变回"普通列"
============================================
'''

# 按年份和会员ID分组聚合
rfm_gb = data.groupby(['year', '会员ID']).agg({
    'date_diff': 'min',   # R：取最小 date_diff（最近一次消费离年底最近）
    '订单号': 'count',    # F：统计订单次数
    '订单金额': 'sum'     # M：统计消费总金额
}).reset_index()

# 重命名列，方便后续使用
rfm_gb.columns = ['year', '会员ID', 'R', 'F', 'M']

print("===== RFM 聚合结果预览 =====")
print(rfm_gb.head())


===== RFM 聚合结果预览 =====
   year  会员ID    R  F       M
0  2015   267  197  2   105.0
1  2015   282  251  1    29.7
2  2015   283  340  1  5398.0
3  2015   343  300  1   118.0
4  2015   525   37  3   213.0


In [40]:
'''
============================================
RFM 评分：对 R/F/M 分别划分等级（1~3分）
============================================

评分规则说明：
  R（Recency）— 最近一次消费距今多少天
     天数越少 → 客户越活跃 → 分数越高
     0~79天   → 3分（高活跃）
     80~255天 → 2分（一般）
     256~365天→ 1分（沉睡）

  F（Frequency）— 一年内消费了多少次
     次数越多 → 客户越忠诚 → 分数越高
     1~2次    → 1分
     3~5次    → 2分
     5次以上  → 3分（忠实客户）

  M（Monetary）— 一年内消费总金额
     金额越大 → 客户价值越高 → 分数越高
     1~69元       → 1分
     70~1199元    → 2分
     1200~206252元→ 3分（高价值客户）

  pd.cut(数据, bins=区间边界, labels=分数) 的用法：
    - bins = [-1, 79, 255, 365] 表示 3 个区间：(-1,79], (79,255], (255,365]
    - labels = [3, 2, 1] 分别对应上面的 3 个区间
    注意：区间是 左开右闭 的，例如 (-1, 79] 表示 大于-1 且 小于等于79
============================================
'''

# ===== R 评分：最近一次消费时间（天数），越小越好 =====
r_bins = [-1, 79, 255, 365]  # 三个区间边界
# 区间划分：(-1, 79] → (79, 255] → (255, 365]
rfm_gb['R_label'] = pd.cut(rfm_gb['R'], bins=r_bins, labels=[3, 2, 1])
# 注意 labels=[3,2,1] 而不是 [1,2,3]：
#   R 越小越好，所以 R ∈ [0,79] → 得 3 分（最高分）

# ===== F 评分：消费次数，越大越好 =====
f_bins = [0, 2, 5, 130]
# 区间划分：(0, 2] → (2, 5] → (5, 130]
rfm_gb['F_label'] = pd.cut(rfm_gb['F'], bins=f_bins, labels=[1, 2, 3])
# F 越大越好，所以 F ≥ 6 → 得 3 分（最高分）

# ===== M 评分：消费金额，越大越好 =====
m_bins = [1, 69, 1199, 206252]
# 区间划分：(1, 69] → (69, 1199] → (1199, 206252]
rfm_gb['M_label'] = pd.cut(rfm_gb['M'], bins=m_bins, labels=[1, 2, 3])
# M 越大越好，所以 M ≥ 1200 → 得 3 分（最高分）

print(rfm_gb.head())


   year  会员ID    R  F       M R_label F_label M_label
0  2015   267  197  2   105.0       2       1       2
1  2015   282  251  1    29.7       2       1       1
2  2015   283  340  1  5398.0       1       1       3
3  2015   343  300  1   118.0       1       1       2
4  2015   525   37  3   213.0       3       2       2


In [41]:
'''
============================================
第7步：组合 RFM 标签
============================================
将 R、F、M 三个分数拼成一个 3 位数标签。

例如 R_label=3, F_label=1, M_label=2 → RFM_label = "312"
  R=3 表示最近刚买（活跃）
  F=1 表示买得少（不常来）
  M=2 表示消费中等

astype(str) 将数字转为字符串，
然后用 + 拼接起来。
============================================
'''

# 将 R、F、M 的分数拼接成 3 位标签
rfm_gb['RFM_label'] = (
    rfm_gb['R_label'].astype(str)
    + rfm_gb['F_label'].astype(str)
    + rfm_gb['M_label'].astype(str)
)

print("===== 添加 RFM_label 后的数据 =====")
print(rfm_gb.head())


===== 添加 RFM_label 后的数据 =====
   year  会员ID    R  F       M R_label F_label M_label RFM_label
0  2015   267  197  2   105.0       2       1       2       212
1  2015   282  251  1    29.7       2       1       1       211
2  2015   283  340  1  5398.0       1       1       3       113
3  2015   343  300  1   118.0       1       1       2       112
4  2015   525   37  3   213.0       3       2       2       322


In [42]:
'''
============================================
第8步：导出结果到 Excel
============================================
将计算好的 RFM 数据保存为 Excel 文件，
方便后续用 Excel 或其他工具查看和分析。

.to_excel(file, index=False)：
  index=False 表示不写入行号（0,1,2...）
============================================
'''

rfm_gb.to_excel('rfm_gb.xlsx', index=False)

print("✅ 已导出到 rfm_gb.xlsx")
print(f"共 {len(rfm_gb)} 条记录")


✅ 已导出到 rfm_gb.xlsx
共 148591 条记录


In [43]:
'''
============================================
数据可视化准备：按 RFM 等级 + 年份 统计人数
============================================

为下一步画图准备数据。我们要看：
  每一种 RFM 客户类型（如 111、112...333），
  在每一年（2015~2018）各有多少人。

.groupby(['A', 'B'])['C'].count() 的含义：
  - 按 A 和 B 两列分组
  - 对每组内的 C 列计数
  - 结果就是：每种 (A, B) 组合有多少条记录
============================================
'''

# ===== 按 RFM 等级 + 年份 分组统计会员数量 =====
display_data = rfm_gb.groupby(['RFM_label', 'year'], as_index=False)['会员ID'].count()
# as_index=False：分组键不作为索引，而是保留为普通列
# 结果列的默认名字是 '会员ID'（因为我们对 '会员ID' 做了 count）

display_data.columns = ['RFM_label', 'year', 'count']
# 重命名列，让含义更清晰

display_data['RFM_label'] = display_data['RFM_label'].astype(np.int32)
# 转为整数类型，方便后续排序和索引

print("===== 每种RFM类型在各年份的会员数量 =====")
print(display_data)


===== 每种RFM类型在各年份的会员数量 =====
    RFM_label  year  count
0         111  2015   2180
1         111  2016   1498
2         111  2017   3169
3         111  2018   2271
4         112  2015   3811
..        ...   ...    ...
83        332  2018     24
84        333  2015     15
85        333  2016     28
86        333  2017     87
87        333  2018    355

[88 rows x 3 columns]


In [47]:
'''
============================================
RFM 等级分布 3D 柱状图
============================================

使用 pyecharts 的 Bar3D 绘制交互式 3D 柱状图。

数据展示逻辑：
  - X 轴：RFM_label（客户类型，如 111、112、…、333）
    RFM_label 的 3 位数分别是 R分数·F分数·M分数
    例如 311 → R=3(活跃), F=1(消费少), M=1(消费低)

  - Y 轴：年份（2015、2016、2017、2018）
  - Z 轴（柱子高度）：该 RFM 类型在该年份的会员数量

pyecharts Bar3D 使用要点（v2.x）：
  数据格式：[[x索引, y索引, z值], ...]
  xaxis3d_opts / yaxis3d_opts 放在 add() 方法中
============================================
'''

# ===== 第1步：整理数据格式 =====
# display_data 来自上一个单元格，包含：RFM_label, year, count

x_data = sorted(display_data['RFM_label'].unique())  # X轴：所有RFM类型
y_data = sorted(display_data['year'].unique())  # Y轴：所有年份

# 转为标准 Python int（numpy int 在某些库中可能不兼容）
x_list = [int(x) for x in x_data]
y_list = [int(y) for y in y_data]

data_3d = []
for _, row in display_data.iterrows():
    x_idx = x_list.index(int(row['RFM_label']))
    y_idx = y_list.index(int(row['year']))
    data_3d.append([x_idx, y_idx, int(row['count'])])

print(f"X轴（RFM类型）共 {len(x_list)} 种")
print(f"Y轴（年份）: {y_list}")
print(f"3D数据点共 {len(data_3d)} 个")

# ===== 第2步：绘制 3D 柱状图 =====
bar3d = Bar3D()
bar3d.add(
    series_name="会员数量",
    data=data_3d,
    shading="realistic",
    label_opts=opts.LabelOpts(is_show=False),
    # ★ v2.x 中轴参数放在 add() 里 ★
    xaxis3d_opts=opts.Axis3DOpts(
        data=[str(x) for x in x_list], name="RFM等级", type_="category"
    ),
    yaxis3d_opts=opts.Axis3DOpts(
        data=[str(y) for y in y_list], name="年份", type_="category"
    ),
    zaxis3d_opts=opts.Axis3DOpts(name="会员数量", type_="value"),
)

bar3d.set_global_opts(
    title_opts=opts.TitleOpts(title="RFM等级分布3D柱状图"),
    visualmap_opts=opts.VisualMapOpts(
        max_=int(display_data['count'].max()),
        range_color=[
            "#313695", "#4575b4", "#74add1", "#abd9e9",
            "#fee090", "#fdae61", "#f46d43", "#d73027",
        ],
    ),
)

# 在 notebook 中显示（支持鼠标拖拽旋转、滚轮缩放）
bar3d.render_notebook()
bar3d.render("rfm_3d_bar.html")  # 同时保存为 HTML 文件
print("✅ 3D 柱状图已保存为 rfm_3d_bar.html")

X轴（RFM类型）共 25 种
Y轴（年份）: [2015, 2016, 2017, 2018]
3D数据点共 88 个
✅ 3D 柱状图已保存为 rfm_3d_bar.html
